In [1]:
import pandas as pd
import numpy as np
import re
import calendar
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
HMIS_PROCESSED_DIR        = PROCESSED_DIR / "Hospital HMIS Dataset for Healthcare Analytics"

# =========================================================
# LOAD CLEANED HMIS TABLES
# (dates get saved as text in CSV, so we re-parse them here)
# =========================================================
cleaned_dataframes = {}
for name in ['department','patient','employee','disease','insurance_provider','drug_manufacturer',
             'doctor','ward','drug','patient_insurance','bed','drug_inventory','staff_assignment',
             'admission','diagnostic_test','patient_diagnostic','prescription','billing','billing_detail']:
    cleaned_dataframes[name] = pd.read_csv(HMIS_PROCESSED_DIR / f"{name}.csv")

date_cols_by_table = {
    'patient': ['date_of_birth'], 'employee': ['date_of_joining'],
    'admission': ['admission_date', 'discharge_date'],
    'patient_insurance': ['policy_start_date', 'policy_end_date'],
    'drug_inventory': ['last_restock_date'], 'patient_diagnostic': ['test_date'],
    'billing': ['bill_date'],
}
for table, cols in date_cols_by_table.items():
    for c in cols:
        cleaned_dataframes[table][c] = pd.to_datetime(cleaned_dataframes[table][c])

admission = cleaned_dataframes['admission']
patient = cleaned_dataframes['patient']
department = cleaned_dataframes['department']
ward = cleaned_dataframes['ward']
bed = cleaned_dataframes['bed']
disease = cleaned_dataframes['disease']
employee = cleaned_dataframes['employee']
doctor = cleaned_dataframes['doctor']
billing = cleaned_dataframes['billing']
patient_insurance = cleaned_dataframes['patient_insurance']
insurance_provider = cleaned_dataframes['insurance_provider']
diagnostic_test = cleaned_dataframes['diagnostic_test']
patient_diagnostic = cleaned_dataframes['patient_diagnostic']
prescription = cleaned_dataframes['prescription']
staff_assignment = cleaned_dataframes['staff_assignment']
drug = cleaned_dataframes['drug']
drug_inventory = cleaned_dataframes['drug_inventory']
drug_manufacturer = cleaned_dataframes['drug_manufacturer']

print("HMIS models a single hospital (no hospital_id column anywhere).")
print("hospital_id=1 / hospital_name='HMIS Hospital' used as a placeholder throughout.\n")

HMIS models a single hospital (no hospital_id column anywhere).
hospital_id=1 / hospital_name='HMIS Hospital' used as a placeholder throughout.



In [2]:
# =========================================================
# SHARED HELPERS
# =========================================================
def add_date_parts(df, date_col, prefix=""):
    df[f"{prefix}year"] = df[date_col].dt.year
    df[f"{prefix}quarter"] = df[date_col].dt.quarter
    df[f"{prefix}month"] = df[date_col].dt.month
    df[f"{prefix}month_name"] = df[date_col].dt.month_name()
    df[f"{prefix}day_of_week"] = df[date_col].dt.day_name()
    return df


# doctor -> employee_name / specialization lookup, reused by table 1 and table 2
doctor_lookup = doctor.merge(employee[['employee_id', 'employee_name']], on='employee_id', how='left')
doctor_lookup = doctor_lookup[['doctor_id', 'employee_name', 'specialization']].rename(
    columns={'employee_name': 'doctor_name', 'specialization': 'doctor_specialization'})


# =========================================================
# BUILD ADMISSION-LEVEL BASE TABLE
# (used directly for Table 1, and as the source for Tables 2-4)
# =========================================================
adm = admission.copy()

adm = adm.merge(department[['department_id', 'department_name', 'department_type']],
                 on='department_id', how='left')
adm = adm.merge(ward[['ward_id', 'ward_name', 'ward_type']], on='ward_id', how='left')
adm = adm.merge(bed[['bed_id', 'bed_number']], on='bed_id', how='left')
adm = adm.merge(disease[['disease_id', 'disease_name', 'disease_category']], on='disease_id', how='left')

pat_small = patient[['patient_id', 'gender', 'date_of_birth', 'blood_group', 'city']].rename(
    columns={'gender': 'patient_gender'})
adm = adm.merge(pat_small, on='patient_id', how='left')
adm['patient_age_at_admission'] = ((adm['admission_date'] - adm['date_of_birth']).dt.days // 365).astype('Int64')

adm['length_of_stay_days'] = (adm['discharge_date'] - adm['admission_date']).dt.days

# --- readmission_flag: same patient re-admitted within 30 days of a previous discharge ---
adm = adm.sort_values(['patient_id', 'admission_date']).reset_index(drop=True)
adm['prev_discharge_date'] = adm.groupby('patient_id')['discharge_date'].shift(1)
adm['readmission_gap_days'] = (adm['admission_date'] - adm['prev_discharge_date']).dt.days
adm['readmission_flag'] = ((adm['readmission_gap_days'].notna()) & (adm['readmission_gap_days'] <= 30)).astype(int)
adm = adm.drop(columns=['prev_discharge_date'])

# --- primary_doctor: proxy = doctor who ordered the EARLIEST diagnostic test in that admission ---
# ~70% of admissions have a diagnostic record; the rest are labelled "No diagnostic record".
pdx_sorted = patient_diagnostic.sort_values(['admission_id', 'test_date'])
primary_doc = pdx_sorted.drop_duplicates('admission_id', keep='first')[['admission_id', 'doctor_id']]
primary_doc = primary_doc.merge(doctor_lookup, on='doctor_id', how='left').rename(
    columns={'doctor_id': 'primary_doctor_id', 'doctor_name': 'primary_doctor_name',
             'doctor_specialization': 'primary_doctor_specialization'})
adm = adm.merge(primary_doc, on='admission_id', how='left')
adm['primary_doctor_name'] = adm['primary_doctor_name'].fillna('No diagnostic record')
adm['primary_doctor_specialization'] = adm['primary_doctor_specialization'].fillna('Unknown')

# --- test / prescription intensity counts ---
dx_count = patient_diagnostic.groupby('admission_id').size().rename('diagnostic_test_count')
presc_count = prescription.groupby('admission_id').size().rename('prescription_count')
distinct_drug = prescription.groupby('admission_id')['drug_id'].nunique().rename('distinct_drug_count')
adm = adm.merge(dx_count, on='admission_id', how='left')
adm = adm.merge(presc_count, on='admission_id', how='left')
adm = adm.merge(distinct_drug, on='admission_id', how='left')
for c in ['diagnostic_test_count', 'prescription_count', 'distinct_drug_count']:
    adm[c] = adm[c].fillna(0).astype(int)

# --- billing ---
bill_small = billing[['admission_id', 'total_amount', 'insurance_covered_amount',
                       'patient_payable_amount', 'payment_status', 'payment_mode']].rename(
    columns={'total_amount': 'total_bill_amount'})
adm = adm.merge(bill_small, on='admission_id', how='left')

# --- insurance provider: match a policy whose start/end window covers admission_date ---
# Only ~24% of admissions land inside an active policy window; the rest = "No active policy on file".
pi = patient_insurance.merge(
    insurance_provider[['insurance_provider_id', 'provider_name', 'provider_type']],
    on='insurance_provider_id', how='left')
adm_pi = adm[['admission_id', 'patient_id', 'admission_date']].merge(pi, on='patient_id', how='left')
mask = (adm_pi['admission_date'] >= adm_pi['policy_start_date']) & \
       (adm_pi['admission_date'] <= adm_pi['policy_end_date'])
adm_pi_matched = adm_pi[mask].sort_values(['admission_id', 'policy_start_date']).drop_duplicates(
    'admission_id', keep='last')
adm_pi_matched = adm_pi_matched[['admission_id', 'provider_name', 'provider_type']].rename(
    columns={'provider_name': 'insurance_provider_name', 'provider_type': 'insurance_provider_type'})
adm = adm.merge(adm_pi_matched, on='admission_id', how='left')
adm['insurance_provider_name'] = adm['insurance_provider_name'].fillna('No active policy on file')
adm['insurance_provider_type'] = adm['insurance_provider_type'].fillna('Unknown')

adm = add_date_parts(adm, 'admission_date')
adm.insert(0, 'hospital_id', 1)
adm.insert(1, 'hospital_name', 'HMIS Hospital')

In [3]:
# =========================================================
# TABLE 1: hospital_overview_dataset
# Grain = one row per admission
# =========================================================
hospital_overview_dataset = adm[[
    'hospital_id', 'hospital_name', 'admission_id', 'patient_id',
    'department_id', 'department_name', 'department_type',
    'ward_id', 'ward_name', 'ward_type', 'bed_id', 'bed_number',
    'admission_date', 'discharge_date', 'length_of_stay_days',
    'admission_type',
    'patient_age_at_admission', 'patient_gender', 'blood_group', 'city',
    'disease_name', 'disease_category',
    'primary_doctor_id', 'primary_doctor_name', 'primary_doctor_specialization',
    'diagnostic_test_count', 'prescription_count', 'distinct_drug_count',
    'total_bill_amount', 'insurance_covered_amount', 'patient_payable_amount',
    'payment_status', 'payment_mode',
    'insurance_provider_name', 'insurance_provider_type',
    'readmission_flag', 'readmission_gap_days',
    'year', 'quarter', 'month', 'month_name', 'day_of_week',
]].copy()


# =========================================================
# TABLE 2: patient_flow_dataset
# Grain = one row per flow event (Admission / Discharge / Diagnostic Test)
# There is no literal transfer/movement log in HMIS, so every admission
# contributes an Admission event + a Discharge event, and every diagnostic
# test contributes a Diagnostic Test event (a genuine move to Radiology/Pathology).
# =========================================================
flow_base_cols = ['admission_id', 'patient_id', 'department_id', 'department_name',
                   'ward_id', 'ward_name', 'ward_type', 'bed_id', 'admission_type',
                   'disease_name', 'disease_category', 'patient_gender', 'patient_age_at_admission']

admission_events = adm[flow_base_cols + ['admission_date']].rename(columns={'admission_date': 'event_date'})
admission_events['event_type'] = 'Admission'
admission_events['length_of_stay_days'] = pd.NA

discharge_events = adm[flow_base_cols + ['discharge_date', 'length_of_stay_days']].rename(
    columns={'discharge_date': 'event_date'})
discharge_events['event_type'] = 'Discharge'

dx_events = patient_diagnostic.merge(
    diagnostic_test[['test_id', 'test_name', 'test_category', 'department_id']], on='test_id', how='left')
dx_events = dx_events.merge(department[['department_id', 'department_name']], on='department_id', how='left')
dx_events = dx_events.merge(doctor_lookup, on='doctor_id', how='left')
dx_events = dx_events.merge(
    adm[['admission_id', 'patient_id', 'admission_type', 'disease_name', 'disease_category',
         'patient_gender', 'patient_age_at_admission']],
    on='admission_id', how='left')
dx_events = dx_events.rename(columns={'test_date': 'event_date'})
dx_events['event_type'] = 'Diagnostic Test'
dx_events['ward_id'] = pd.NA
dx_events['ward_name'] = pd.NA
dx_events['ward_type'] = pd.NA
dx_events['bed_id'] = pd.NA
dx_events['length_of_stay_days'] = pd.NA

flow_cols = flow_base_cols + ['event_type', 'event_date', 'length_of_stay_days',
                               'doctor_id', 'doctor_name', 'doctor_specialization',
                               'test_name', 'test_category', 'result_status']

for block in [admission_events, discharge_events, dx_events]:
    for c in flow_cols:
        if c not in block.columns:
            block[c] = pd.NA

patient_flow_dataset = pd.concat(
    [admission_events[flow_cols], discharge_events[flow_cols], dx_events[flow_cols]],
    ignore_index=True)

patient_flow_dataset.insert(0, 'flow_event_id', range(1, len(patient_flow_dataset) + 1))
patient_flow_dataset.insert(1, 'hospital_id', 1)
patient_flow_dataset.insert(2, 'hospital_name', 'HMIS Hospital')

patient_flow_dataset['event_date'] = pd.to_datetime(patient_flow_dataset['event_date'])
patient_flow_dataset = add_date_parts(patient_flow_dataset, 'event_date')
patient_flow_dataset = patient_flow_dataset.sort_values(['event_date', 'admission_id']).reset_index(drop=True)


# =========================================================
# TABLE 3: department_analytics_dataset
# Grain = one row per department per calendar day (full spine, 2020-01-01..max discharge_date)
# so trend charts in Tableau have no gaps.
# =========================================================
all_dates = pd.date_range(admission['admission_date'].min(), admission['discharge_date'].max(), freq='D')
all_depts = department['department_id'].tolist()
spine = pd.MultiIndex.from_product([all_dates, all_depts], names=['date', 'department_id']).to_frame(index=False)
spine = spine.merge(department[['department_id', 'department_name', 'department_type']],
                     on='department_id', how='left')

adm_daily = adm.groupby(['admission_date', 'department_id']).size().rename('admissions_count') \
    .reset_index().rename(columns={'admission_date': 'date'})
disch_daily = adm.groupby(['discharge_date', 'department_id']).size().rename('discharges_count') \
    .reset_index().rename(columns={'discharge_date': 'date'})
spine = spine.merge(adm_daily, on=['date', 'department_id'], how='left')
spine = spine.merge(disch_daily, on=['date', 'department_id'], how='left')
spine[['admissions_count', 'discharges_count']] = spine[['admissions_count', 'discharges_count']].fillna(0).astype(int)

# estimated_census: running (admissions - discharges) per department = proxy for "patients in dept that day"
spine = spine.sort_values(['department_id', 'date'])
spine['estimated_census'] = (spine['admissions_count'] - spine['discharges_count']) \
    .groupby(spine['department_id']).cumsum().clip(lower=0)

dept_beds = ward.groupby('department_id')['total_beds'].sum().rename('total_beds').reset_index()
spine = spine.merge(dept_beds, on='department_id', how='left')
spine['total_beds'] = spine['total_beds'].fillna(0).astype(int)
spine['bed_occupancy_rate_pct'] = (spine['estimated_census'] / spine['total_beds'].astype(float).replace(0, np.nan) * 100).round(2)

readm_daily = adm[adm['readmission_flag'] == 1].groupby(['admission_date', 'department_id']).size() \
    .rename('readmission_count').reset_index().rename(columns={'admission_date': 'date'})
spine = spine.merge(readm_daily, on=['date', 'department_id'], how='left')
spine['readmission_count'] = spine['readmission_count'].fillna(0).astype(int)
spine['readmission_rate_pct'] = (spine['readmission_count'] / spine['admissions_count'].astype(float).replace(0, np.nan) * 100).round(2)

los_daily = adm.groupby(['discharge_date', 'department_id'])['length_of_stay_days'].mean() \
    .rename('avg_length_of_stay_days').reset_index().rename(columns={'discharge_date': 'date'})
spine = spine.merge(los_daily, on=['date', 'department_id'], how='left')

bill_daily = adm.groupby(['admission_date', 'department_id'])['total_bill_amount'].agg(['sum', 'mean']) \
    .reset_index().rename(columns={'admission_date': 'date', 'sum': 'total_bill_amount', 'mean': 'avg_bill_amount'})
spine = spine.merge(bill_daily, on=['date', 'department_id'], how='left')
spine[['total_bill_amount', 'avg_bill_amount']] = spine[['total_bill_amount', 'avg_bill_amount']].fillna(0).round(2)

dx_daily = patient_diagnostic.merge(diagnostic_test[['test_id', 'department_id']], on='test_id', how='left')
dx_daily = dx_daily.groupby(['test_date', 'department_id']).size().rename('diagnostic_test_count') \
    .reset_index().rename(columns={'test_date': 'date'})
spine = spine.merge(dx_daily, on=['date', 'department_id'], how='left')
spine['diagnostic_test_count'] = spine['diagnostic_test_count'].fillna(0).astype(int)

# static staff headcount proxy (employee.department_id) -- NOT a true daily on-duty count
staff_counts = employee[employee['role'].isin(['Doctor', 'Nurse'])] \
    .groupby(['department_id', 'role']).size().unstack(fill_value=0).reset_index()
staff_counts = staff_counts.rename(columns={'Doctor': 'doctor_headcount', 'Nurse': 'nurse_headcount'})
spine = spine.merge(staff_counts, on='department_id', how='left')
for c in ['doctor_headcount', 'nurse_headcount']:
    spine[c] = spine[c].fillna(0).astype(int)
spine['staff_to_patient_ratio'] = ((spine['doctor_headcount'] + spine['nurse_headcount']) /
                                    spine['estimated_census'].astype(float).replace(0, np.nan)).round(3)

# department_efficiency_score (0-100 composite, documented formula):
#   40% occupancy + 30% LOS(inverse, capped at 15 days = worst observed) + 30% readmission(inverse)
MAX_REASONABLE_LOS = 15  # observed max LOS in this dataset
occupancy_score = spine['bed_occupancy_rate_pct'].clip(upper=100)
los_score = (100 - (spine['avg_length_of_stay_days'] / MAX_REASONABLE_LOS * 100)).clip(lower=0, upper=100)
readmission_score = (100 - spine['readmission_rate_pct']).clip(lower=0, upper=100)
spine['department_efficiency_score'] = (0.4 * occupancy_score + 0.3 * los_score + 0.3 * readmission_score).round(2)

spine = add_date_parts(spine, 'date')
spine.insert(0, 'hospital_id', 1)
spine.insert(1, 'hospital_name', 'HMIS Hospital')

department_analytics_dataset = spine[[
    'hospital_id', 'hospital_name', 'date', 'year', 'quarter', 'month', 'month_name', 'day_of_week',
    'department_id', 'department_name', 'department_type',
    'total_beds', 'admissions_count', 'discharges_count', 'estimated_census',
    'bed_occupancy_rate_pct', 'readmission_count', 'readmission_rate_pct',
    'avg_length_of_stay_days', 'total_bill_amount', 'avg_bill_amount',
    'diagnostic_test_count', 'doctor_headcount', 'nurse_headcount',
    'staff_to_patient_ratio', 'department_efficiency_score',
]].sort_values(['department_id', 'date']).reset_index(drop=True)


# =========================================================
# TABLE 4: resource_utilization_dataset
# Grain = department + date + resource_type. Only "Bed" is genuinely daily;
# "Staff" and "Drug Inventory" are current snapshots (no date history exists
# in HMIS for them) -- kept in the same long table, flagged via resource_type.
# =========================================================
# --- Block A: Bed (daily, only depts that actually have wards) ---
bed_block = department_analytics_dataset[department_analytics_dataset['total_beds'] > 0][[
    'hospital_id', 'hospital_name', 'date', 'year', 'quarter', 'month', 'month_name', 'day_of_week',
    'department_id', 'department_name',
    'total_beds', 'estimated_census', 'bed_occupancy_rate_pct',
]].rename(columns={'total_beds': 'total_units_available',
                    'estimated_census': 'units_in_use',
                    'bed_occupancy_rate_pct': 'utilization_rate_pct'})
bed_block['resource_type'] = 'Bed'

# --- Block B: Staff (static snapshot, no date column exists in staff_assignment) ---
staff_ward = staff_assignment.merge(ward[['ward_id', 'ward_name', 'department_id']], on='ward_id', how='left')
staff_ward = staff_ward.merge(department[['department_id', 'department_name']], on='department_id', how='left')
staff_ward = staff_ward.merge(employee[['employee_id', 'role']], on='employee_id', how='left')

staff_block = staff_ward.groupby(['department_id', 'department_name', 'ward_id', 'ward_name', 'shift']) \
    .agg(units_in_use=('employee_id', 'nunique')).reset_index()
# NOTE: staff_assignment.csv only ever carries Nurse/Technician roles (doctors are not
# ward-assigned in this dataset) -- pivot dynamically instead of hardcoding role names.
role_pivot = staff_ward.groupby(['ward_id', 'shift', 'role']).size().unstack(fill_value=0).reset_index()
role_pivot.columns = [c if c in ('ward_id', 'shift') else f"{str(c).lower()}_count" for c in role_pivot.columns]
staff_block = staff_block.merge(role_pivot, on=['ward_id', 'shift'], how='left')
staff_block['resource_type'] = 'Staff'
staff_block['date'] = pd.NaT
staff_block['hospital_id'] = 1
staff_block['hospital_name'] = 'HMIS Hospital'

# --- Block C: Drug Inventory (snapshot as of last_restock_date; assigned to Pharmacy dept as an assumption) ---
drug_block = drug_inventory.merge(drug[['drug_id', 'drug_name', 'drug_category', 'manufacturer_id']],
                                   on='drug_id', how='left')
drug_block = drug_block.merge(drug_manufacturer[['manufacturer_id', 'manufacturer_name', 'reliability_rating']],
                               on='manufacturer_id', how='left')
drug_block['shortage_flag'] = (drug_block['current_stock'] < drug_block['reorder_level']).astype(int)
drug_block['resource_type'] = 'Drug Inventory'
drug_block['date'] = drug_block['last_restock_date']
pharmacy_row = department[department['department_name'] == 'Pharmacy'].iloc[0]
drug_block['department_id'] = pharmacy_row['department_id']
drug_block['department_name'] = pharmacy_row['department_name']
drug_block['hospital_id'] = 1
drug_block['hospital_name'] = 'HMIS Hospital'
drug_block = drug_block.rename(columns={'current_stock': 'units_in_use', 'reorder_level': 'reorder_level_threshold'})

resource_cols = [
    'hospital_id', 'hospital_name', 'resource_type', 'date',
    'department_id', 'department_name', 'ward_id', 'ward_name', 'shift',
    'total_units_available', 'units_in_use', 'utilization_rate_pct',
    'doctor_count', 'nurse_count', 'technician_count', 'pharmacist_count', 'admin_count',
    'drug_id', 'drug_name', 'drug_category', 'reorder_level_threshold',
    'shortage_flag', 'manufacturer_name', 'reliability_rating',
]
for block in [bed_block, staff_block, drug_block]:
    for c in resource_cols:
        if c not in block.columns:
            block[c] = pd.NA

resource_utilization_dataset = pd.concat(
    [bed_block[resource_cols], staff_block[resource_cols], drug_block[resource_cols]],
    ignore_index=True)
resource_utilization_dataset.insert(0, 'resource_utilization_id',
                                     range(1, len(resource_utilization_dataset) + 1))

C:\Users\sr189\AppData\Local\Temp\ipykernel_21924\116049993.py:224: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  resource_utilization_dataset = pd.concat(


In [4]:
# =========================================================
# QUICK SANITY CHECKS BEFORE SAVING
# =========================================================
print("hospital_overview_dataset:", hospital_overview_dataset.shape)
print("patient_flow_dataset:", patient_flow_dataset.shape)
print("department_analytics_dataset:", department_analytics_dataset.shape)
print("resource_utilization_dataset:", resource_utilization_dataset.shape)
assert hospital_overview_dataset['admission_id'].is_unique
assert hospital_overview_dataset.shape[0] == 45000
print("\nAll sanity checks passed.")


# =========================================================
# SAVE FINAL TABLES
# =========================================================
hospital_overview_dataset.to_csv(PROCESSED_DIR / "hospital_overview_dataset.csv", index=False)
patient_flow_dataset.to_csv(PROCESSED_DIR / "patient_flow_dataset.csv", index=False)
department_analytics_dataset.to_csv(PROCESSED_DIR / "department_analytics_dataset.csv", index=False)
resource_utilization_dataset.to_csv(PROCESSED_DIR / "resource_utilization_dataset.csv", index=False)
print("Saved all 4 final tables to", PROCESSED_DIR)

hospital_overview_dataset: (45000, 42)
patient_flow_dataset: (153269, 30)
department_analytics_dataset: (24244, 26)
resource_utilization_dataset: (13549, 25)

All sanity checks passed.
Saved all 4 final tables to C:\Users\sr189\OneDrive\Desktop\Github infosys\m1&m2 shivam\data\processed
